# P vs NP and the 3-SAT phase transition

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/p_vs_np_sat)

Is every problem whose solutions are easy to check also easy to solve? That is the P vs NP question, open since 1971. Random 3-SAT makes the asymmetry concrete: checking a proposed assignment is one gather and two reductions, while exhaustive search evaluates all $2^n$ assignments at once as one packed bit tensor, whose size doubles with every variable. Sweeping the clause/variable ratio $\alpha$ shows the satisfiability phase transition near $\alpha_c \approx 4.27$ and the search cost peaking inside the transition window.

**Vector form**

$$F(\mathbf x) = \bigwedge_{m=1}^{M} \big(\ell_{m1} \lor \ell_{m2} \lor \ell_{m3}\big),\quad \ell_{mk} = x_{v_{mk}} \oplus s_{mk},\qquad \mathbb E\,\#\{\mathbf x : F(\mathbf x) = 1\} = 2^n \left(\tfrac78\right)^{M}$$

**Index (tensor) form**

$$\ell_{bmk} = x_{b\,v_{bmk}} \oplus s_{bmk},\quad F_b = \bigwedge_m \bigvee_k \ell_{bmk}\qquad T^{F}_{ba} = \bigwedge_m \bigvee_k \big(T_{v_{bmk}\,a} \oplus s_{bmk}\big),\quad a = 0,\dots,2^n-1$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $v_{bmk},\ s_{bmk}$ | `random_3sat(n, M, B, gen, dev)` | (B, M, 3) | $B$ formulas with $M$ clauses of 3 distinct variables, each negated with probability $\frac12$ |
| $\ell_{bmk} = x_{b\,v_{bmk}} \oplus s_{bmk}$ | `torch.gather(x, -1, var.flatten(-2)) ^ neg` | (B, M, 3) bool | every literal of every formula in one gather |
| $\bigwedge_m \bigvee_k$ | `lit.any(dim=-1).all(dim=-1)` | (B,) bool | the verifier: $3M$ lookups, polynomial in $n$ |
| $T_{ja}$ | `pack(all_assignments(n).T)` | (n, 2^n/32) int64 | truth table of $x_j$ over all $2^n$ assignments, 32 assignments per integer |
| $T_{v_1 a}\oplus s_1 \lor T_{v_2 a}\oplus s_2 \lor T_{v_3 a}\oplus s_3$ | `lit[:, 0] \| lit[:, 1] \| lit[:, 2]` | (B, 2^n/32) | one clause evaluated at all $2^n$ assignments with bitwise OR (XOR with all ones negates) |
| $T^F_{ba}$ | `levels & clause[:, None]` | (B, n+1, 2^n/32) | AND in the clauses one by one; level $d$ only collects clauses whose variables are among the first $d$, which gives the backtracking tree |
| $P_{\mathrm{sat}}(\alpha)$ | `res['sat'].double().mean(dim=0)` | (8n+1,) | nested formulas (prefixes of one clause list) give every $\alpha = M/n$ in one pass and make $P_{\mathrm{sat}}$ non-increasing |

P vs NP asks whether every decision problem whose yes-answers have certificates that can be checked in polynomial time (the class NP) can also be solved in polynomial time (the class P). Cook (1971) and Levin (1973) proved that satisfiability is NP-complete: a polynomial-time algorithm for 3-SAT would give one for every problem in NP. It is unknown whether one exists. Most researchers expect P $\ne$ NP, but there is no proof, and known barriers (relativisation, natural proofs, algebrisation) rule out whole families of proof strategies. The question is one of the seven Clay Millennium Prize problems and is open. This program proves nothing about it: exhaustive search is merely the obvious algorithm, and the best known 3-SAT algorithms are still exponential (about $1.307^n$ for PPSZ). What it shows is the asymmetry. The verifier is one gather plus an OR and an AND, $O(M)$ work. The search builds the full truth table, one bit per assignment packed 32 per integer, so the tensor has $2^n$ bits and its time doubles with each variable (bottom right; at small $n$ fixed overheads dominate). Random 3-SAT with $M = \alpha n$ clauses has a sharp threshold: Friedgut (1999) proved that the transition becomes sharp as $n \to \infty$, statistical physics (Mézard, Parisi and Zecchina 2002) predicts $\alpha_c \approx 4.267$, and Ding, Sly and Sun (2015) proved the predicted threshold for $k$-SAT with large $k$. For $k = 3$ the value 4.267 is still a conjecture; what is proven is that random formulas are satisfiable with high probability below $\alpha = 3.52$ and unsatisfiable above 4.49. At small $n$ the 50% crossing lies above 4.27 and drifts down as $n$ grows (finite-size scaling). Tests check that the verifier agrees with brute force on every assignment, that one clause removes exactly $1/8$ of all assignments, that the mean number of solutions equals the first moment $2^n (7/8)^M$, that $P_{\mathrm{sat}}$ is monotone and collapses near the threshold, and that the window narrows with $n$. The search-cost panel counts the nodes of a depth-first backtracking search that fixes $x_0, x_1, \ldots$ in turn and backs up as soon as a fully assigned clause is false. Below the threshold it finds a solution quickly, far above it short contradictions prune the tree early, and the hardest formulas sit in the transition window (for this simple search at $n = 16$ near $\alpha \approx 5$; solvers with unit propagation peak near 4.3). Try more variables to watch the curves sharpen, or more formulas for smoother curves.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/p_vs_np_sat/main.py
"""Main loop: parameters → random formulas → exhaustive search sweep → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors

THRESHOLD = 4.267  # conjectured satisfiability threshold of random 3-SAT (Mézard–Parisi–Zecchina 2002)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_max = 12 if fast_mode() else params["n_max"]
    batch = 60 if fast_mode() else params["instances"]
    gen = torch.Generator().manual_seed(params["seed"])
    n_values = [max(n_max - 8, 6), n_max - 4, n_max]  # n ≥ 5: at least one 32-bit word

    # 1. satisfiable fraction for every clause count M = 0 … 8n, for three sizes n
    p_sat, alpha = {}, {}
    for n in n_values:
        var, neg = tensors.random_3sat(n, 8 * n, batch, gen, dev)
        record = [round(a * n) for a in torch.arange(0, 8.01, 0.5).tolist()] if n == n_max else None
        res = solve.sweep(var, neg, n, record=record, keep=int(n == n_max))
        p_sat[n] = res["sat"].double().mean(dim=0).cpu()  # (8n + 1,)
        alpha[n] = torch.arange(8 * n + 1) / n
        print(
            f"n = {n:2d}: {batch} formulas × 2^{n} = {2**n:,} assignments each; P(sat) = 1/2 at α = M/n ≈ "
            f"{solve.crossing(p_sat[n], n):.2f} (90% → 10% window {solve.crossing(p_sat[n], n, 0.9):.2f} – "
            f"{solve.crossing(p_sat[n], n, 0.1):.2f})"
        )
        if n == n_max:
            maps, record_max = res["maps"].cpu(), record
            nodes = res["nodes"].double().median(dim=0).values.cpu()  # (len(record),)
    peak = record_max[int(nodes.argmax())] / n_max
    print(f"conjectured threshold for n → ∞: α_c ≈ {THRESHOLD} (finite n crosses higher and drifts down towards it)")
    print(
        f"backtracking search (n = {n_max}): median {nodes[0]:.0f} nodes at α = 0, peak {nodes.max():.0f} at "
        f"α ≈ {peak:.2f}, {nodes[-1]:.0f} at α = 8 (easy – hard – easy)"
    )

    # 2. the asymmetry behind P vs NP: checking a certificate versus searching all 2^n assignments
    sizes = list(range(6, n_max + 3, 2))
    cost = solve.timing(sizes, 64, gen, dev)
    for n, tv, ts in zip(cost["n"], cost["verify"], cost["search"]):
        print(f"n = {n:2d}: verify 64 certificates {tv * 1e3:7.3f} ms   search all 2^n assignments {ts * 1e3:9.2f} ms")
    return plot.animate(alpha, p_sat, maps, record_max, nodes, cost, n_max, THRESHOLD, out_dir / "p_vs_np_sat.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/p_vs_np_sat/equation.py
r"""Mathematics → tensor translation.

A 3-SAT formula is an AND of M clauses, each an OR of 3 literals: F(x) = ∧_m ∨_k ℓ_{mk}(x), with the literal
ℓ_{mk}(x) = x_{v_{mk}} ⊕ s_{mk} (variable v, negation sign s).

1. Verifying a certificate (the "NP" in P vs NP): given x, evaluate every literal at once with one gather, then
   OR over k and AND over m. Cost O(M): polynomial.

       ℓ_{bmk} = x_{b, v_{bmk}} ⊕ s_{bmk},   F_b = ∧_m ∨_k ℓ_{bmk}

2. Exhaustive search: evaluate F at all 2^n assignments at once. The 2^n values of x_j form a bit vector (its truth
   table T_j); a clause's table is T_{v1}⊕s1 | T_{v2}⊕s2 | T_{v3}⊕s3 and the formula's table is the AND of the
   clause tables. Packing 32 assignments per integer makes this bitwise tensor algebra, but the tensor still has
   2^n bits: the cost doubles with every extra variable.
"""

import torch

WORD = 32
MASK = (1 << WORD) - 1  # all 32 low bits set: "true for every assignment in this word"


def literals(x: torch.Tensor, var: torch.Tensor, neg: torch.Tensor) -> torch.Tensor:
    r"""ℓ = x_v ⊕ s for every literal.   x: (..., n) bool, var/neg: (..., M, 3) → (..., M, 3) bool"""
    batch = torch.broadcast_shapes(x.shape[:-1], var.shape[:-2])
    m = var.shape[-2]
    x = x.expand(*batch, x.shape[-1])
    idx = var.flatten(-2).expand(*batch, 3 * m)
    return torch.gather(x, -1, idx).view(*batch, m, 3) ^ neg


def verify(x: torch.Tensor, var: torch.Tensor, neg: torch.Tensor) -> torch.Tensor:
    r"""F(x) = ∧_m ∨_k ℓ_{mk}: is x a satisfying assignment?   → (...,) bool. Polynomial: 3M literal lookups."""
    return literals(x, var, neg).any(dim=-1).all(dim=-1)


def pack(bits: torch.Tensor) -> torch.Tensor:
    """(..., 2^n) bool → (..., 2^n / 32) int64; assignment a sits in word a // 32 at bit a % 32."""
    b = bits.reshape(*bits.shape[:-1], -1, WORD).long()
    return (b << torch.arange(WORD, device=bits.device)).sum(dim=-1)  # distinct powers of two: sum = OR


def unpack(words: torch.Tensor, n: int) -> torch.Tensor:
    """(..., W) int64 → (..., 2^n) bool."""
    bits = (words[..., None] >> torch.arange(WORD, device=words.device)) & 1
    return bits.reshape(*words.shape[:-1], 2**n).bool()


def popcount(words: torch.Tensor) -> torch.Tensor:
    """Number of set bits = number of satisfying assignments.   (..., W) → (...,)"""
    return ((words[..., None] >> torch.arange(WORD, device=words.device)) & 1).sum(dim=(-1, -2))


def clause_table(tables: torch.Tensor, var: torch.Tensor, neg: torch.Tensor) -> torch.Tensor:
    r"""Truth table of one clause per formula: T_{v1}⊕s1 | T_{v2}⊕s2 | T_{v3}⊕s3.
    tables: (n, W), var/neg: (B, 3) → (B, W) int64"""
    lit = tables[var] ^ (neg.long() * MASK)[..., None]  # (B, 3, W): XOR with all ones negates
    return lit[:, 0] | lit[:, 1] | lit[:, 2]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/p_vs_np_sat/tensors.py
"""Build the concrete tensors: random 3-SAT formulas, planted formulas and the table of all 2^n assignments."""

import torch


def random_3sat(n: int, m: int, batch: int, gen: torch.Generator, device: torch.device):
    """B random formulas with M clauses of 3 distinct variables, each negated with probability 1/2.

    var: (B, M, 3) int64 variable index, neg: (B, M, 3) bool (True ⇒ the literal is ¬x).
    The first M' clauses of a formula are themselves a random formula with M' clauses, so one formula of length
    M_max gives the whole sweep over the clause/variable ratio α = M'/n ("nested" formulas).
    """
    keys = torch.rand(batch, m, n, generator=gen)
    var = keys.argsort(dim=-1)[..., :3]  # 3 distinct variables per clause, uniformly
    neg = torch.rand(batch, m, 3, generator=gen) < 0.5
    return var.to(device), neg.to(device)


def planted_3sat(x_star: torch.Tensor, m: int, gen: torch.Generator):
    """Random formulas that the hidden assignment x_star (B, n) satisfies: each clause gets one of the 7 sign
    patterns that x_star satisfies (never the one that makes all 3 literals false)."""
    batch, n = x_star.shape
    keys = torch.rand(batch, m, n, generator=gen)
    var = keys.argsort(dim=-1)[..., :3].to(x_star.device)  # (B, M, 3)
    value = torch.gather(x_star[:, None].expand(-1, m, -1), 2, var)  # (B, M, 3) value of each variable
    pattern = torch.randint(1, 8, (batch, m), generator=gen).to(x_star.device)  # 1 … 7, never 0
    true_lit = (pattern[..., None] >> torch.arange(3, device=x_star.device)) & 1  # which literals x_star makes true
    neg = value ^ true_lit.bool()  # literal value = x XOR neg ⇒ choose neg so the chosen literals are true
    return var, neg


def all_assignments(n: int, device: torch.device) -> torch.Tensor:
    """Every assignment a = 0 … 2^n − 1 as bits, x_0 the most significant: (2^n, n) bool.
    In this order a depth-first search that tries x_i = 0 before 1 meets the leaves in increasing a."""
    a = torch.arange(2**n, device=device)
    shift = torch.arange(n - 1, -1, -1, device=device)
    return ((a[:, None] >> shift[None, :]) & 1).bool()


def truth_tables(n: int, device: torch.device) -> torch.Tensor:
    """The column x_j of all_assignments, packed 32 assignments per word: (n, W) int64 with W = 2^n / 32."""
    from .equation import pack

    return pack(all_assignments(n, device).T)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/p_vs_np_sat/solve.py
"""Exhaustive search over all 2^n assignments, clause by clause, for a whole batch of nested formulas at once."""

import math

import torch

from problems._common.progress import steps

from .equation import MASK, WORD, clause_table, unpack
from .tensors import truth_tables


def solution_table(var: torch.Tensor, neg: torch.Tensor, n: int) -> torch.Tensor:
    """Packed truth table of each whole formula: bit a is set iff assignment a satisfies it.   → (B, W)"""
    tables = truth_tables(n, var.device)  # (n, W)
    table = torch.full((var.shape[0], tables.shape[1]), MASK, dtype=torch.int64, device=var.device)
    for m in range(var.shape[1]):  # AND in one clause at a time (all formulas at once)
        table &= clause_table(tables, var[:, m], neg[:, m])
    return table


def first_solution(table: torch.Tensor, n: int) -> torch.Tensor:
    """Smallest satisfying assignment index a* per formula, 2^n if there is none.   (B, W) → (B,)"""
    bits = unpack(table, n)
    return torch.where(bits.any(dim=1), bits.long().argmax(dim=1), 2**n)


def backtrack_nodes(levels: torch.Tensor, n: int) -> torch.Tensor:
    """Nodes a depth-first backtracking search visits before it finds the first solution (or proves there is none).

    levels[:, d] is the packed table of the clauses whose variables all lie among x_0 … x_{d−1}: the partial
    assignments of depth d that no fully assigned clause rejects. The search fixes x_0, x_1, … in turn, tries 0 before
    1 and backs up as soon as a clause is violated, so it meets the leaves in increasing order. A live node of depth d
    covers a block of 2^(n−d) leaves; it is visited iff its block starts at or before the first solution a*.
    levels: (B, n + 1, W) → (B,) int64
    """
    a_star = first_solution(levels[:, n], n).clamp(max=2**n - 1)  # unsatisfiable ⇒ the whole tree is searched
    nodes = torch.zeros(levels.shape[0], dtype=torch.int64, device=levels.device)
    for d in range(n + 1):
        block = 2 ** (n - d)
        if block >= WORD:  # a block starts at bit 0 of every (block / 32)-th word
            live = (levels[:, d, :: block // WORD] & 1).bool()  # (B, 2^d): one entry per node of depth d
        else:
            live = unpack(levels[:, d], n)[:, ::block]
        start = torch.arange(2**d, device=levels.device) * block
        nodes += (live & (start[None, :] <= a_star[:, None])).sum(dim=1)
    return nodes


def sweep(var: torch.Tensor, neg: torch.Tensor, n: int, record: list[int] | None = None, keep: int = 0) -> dict:
    """Add the clauses of nested formulas one by one (M' = 0 … M_max) and track, for every prefix length M':

    sat (B, M_max + 1) bool       is the formula with the first M' clauses satisfiable?
    count (B, M_max + 1) int64    number of satisfying assignments (only for the first `keep` formulas)
    nodes (B, len(record)) int64  backtracking search cost at the prefix lengths listed in `record`
    maps (M_max + 1, 2^n) bool    solution map of formula 0 (only if keep > 0)
    """
    record = record or []
    batch, m_max, _ = var.shape
    tables = truth_tables(n, var.device)  # (n, W)
    levels = torch.full((batch, n + 1, tables.shape[1]), MASK, dtype=torch.int64, device=var.device)
    depth = torch.arange(n + 1, device=var.device)
    sat = torch.ones(batch, m_max + 1, dtype=torch.bool, device=var.device)
    maps = [unpack(levels[0, n], n)] if keep else []
    nodes = []
    if 0 in record:
        nodes.append(backtrack_nodes(levels, n))
    for m in steps(m_max, "clauses"):
        clause = clause_table(tables, var[:, m], neg[:, m])  # (B, W)
        deeper = depth[None, :] > var[:, m].max(dim=1).values[:, None]  # (B, n + 1): levels that see this clause
        levels = torch.where(deeper[..., None], levels & clause[:, None], levels)
        sat[:, m + 1] = (levels[:, n] != 0).any(dim=1)
        if keep:
            maps.append(unpack(levels[0, n], n))
        if m + 1 in record:
            nodes.append(backtrack_nodes(levels, n))
    out = {"sat": sat}
    if record:
        out["nodes"] = torch.stack(nodes, dim=1)
    if keep:
        out["maps"] = torch.stack(maps)
    return out


def crossing(p_sat: torch.Tensor, n: int, level: float = 0.5) -> float:
    """Ratio α = M/n where the (non-increasing) satisfiable fraction first drops below `level`, linearly
    interpolated between neighbouring clause counts."""
    hits = torch.nonzero(p_sat < level)
    if len(hits) == 0:
        return float("nan")  # never drops below `level` within the sweep
    below = int(hits[0])
    p0, p1 = p_sat[below - 1].item(), p_sat[below].item()
    m = below - 1 + (p0 - level) / max(p0 - p1, 1e-12)
    return m / n


def timing(n_values: list[int], batch: int, gen: torch.Generator, device: torch.device) -> dict:
    """Wall time to verify a batch of certificates versus to search all 2^n assignments, at α ≈ 4.27."""
    import time

    from .equation import verify
    from .tensors import random_3sat

    out = {"n": n_values, "verify": [], "search": []}
    var, neg = random_3sat(6, 26, batch, gen, device)
    verify(var[..., 0] > 2, var, neg), solution_table(var, neg, 6)  # warm-up: first calls pay one-off costs
    for n in n_values:
        m = math.ceil(4.27 * n)
        var, neg = random_3sat(n, m, batch, gen, device)
        x = torch.rand(batch, n, generator=gen).to(device) < 0.5
        t0 = time.perf_counter()
        for _ in range(20):
            verify(x, var, neg)
        t1 = time.perf_counter()
        for _ in range(3):
            solution_table(var, neg, n)
        t2 = time.perf_counter()
        out["verify"].append((t1 - t0) / 20)
        out["search"].append((t2 - t1) / 3)
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/p_vs_np_sat/plot.py
"""Solution map of one formula as clauses are added, P(sat) against α, search cost, and verify vs search time."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(alpha, p_sat, maps, record, nodes, cost, n, threshold, path: Path) -> Path:
    fig, ((ax_m, ax_p), (ax_w, ax_t)) = plt.subplots(2, 2, figsize=(10, 6.6))
    image = maps.numpy().reshape(len(maps), 2 ** (n // 2), -1)  # (M + 1, 2^(n/2), 2^(n − n/2)): one cell each
    im = ax_m.imshow(image[0], cmap=ListedColormap(["#f3f4f6", "#2563eb"]), vmin=0, vmax=1, interpolation="nearest")
    ax_m.set_xticks([])
    ax_m.set_yticks([])
    ax_m.grid(False)
    title_m = ax_m.set_title("", fontsize=9)
    ax_m.set_autoscale_on(False)
    few = ax_m.scatter([], [], s=14, color="#2563eb", edgecolors="#1e40af")  # rare survivors drawn larger

    for (size, p), colour in zip(p_sat.items(), ACCENT):
        ax_p.plot(alpha[size].numpy(), p.numpy(), color=colour, lw=1.5, label=f"$n = {size}$")
    ax_p.axvline(threshold, color="#9ca3af", ls="--", lw=1)
    ax_p.text(threshold + 0.08, 0.92, r"$\alpha_c \approx 4.27$", fontsize=8, color="#6b7280")
    ax_p.set_xlabel(r"clauses per variable $\alpha = M/n$")
    ax_p.set_ylabel("fraction satisfiable")
    ax_p.set_title("satisfiability phase transition", fontsize=9)
    ax_p.legend(fontsize=8, loc="lower left")
    cursor_p = ax_p.axvline(0, color="#dc2626", lw=1)

    a_rec = np.array(record) / n
    ax_w.semilogy(a_rec, nodes.numpy(), "o-", color=ACCENT[2], ms=3, lw=1.3)
    ax_w.axvline(threshold, color="#9ca3af", ls="--", lw=1)
    ax_w.set_xlabel(r"$\alpha = M/n$")
    ax_w.set_ylabel("backtracking nodes (median)")
    ax_w.set_title(f"search cost at $n = {n}$: easy – hard – easy", fontsize=9)
    cursor_w = ax_w.axvline(0, color="#dc2626", lw=1)

    sizes = np.array(cost["n"])
    ax_t.semilogy(sizes, np.array(cost["search"]) * 1e3, "o-", color=ACCENT[1], label="search all $2^n$ assignments")
    ax_t.semilogy(sizes, np.array(cost["verify"]) * 1e3, "o-", color=ACCENT[0], label="verify a certificate")
    ax_t.set_xlabel("number of variables $n$")
    ax_t.set_ylabel("time for 64 formulas [ms]")
    ax_t.set_title(r"checking is cheap, exhaustive search doubles per variable", fontsize=9)
    ax_t.legend(fontsize=8, loc="upper left")

    def update(m: int):
        im.set_data(image[m])
        a = m / n
        k = int(image[m].sum())
        rows, cols = np.nonzero(image[m])
        few.set_offsets(np.c_[cols, rows] if k <= 200 else np.empty((0, 2)))
        title_m.set_text(f"one formula, M = {m} clauses (α = {a:.2f}): {k:,} of {2**n:,} assignments satisfy it")
        cursor_p.set_xdata([a, a])
        cursor_w.set_xdata([a, a])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(image), 48), path, fps=5)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 16,
    'instances': 200,
    'seed': 1
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.p_vs_np_sat")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.p_vs_np_sat.main")
problem_dir = Path("problems/maths/p_vs_np_sat")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
